# QniNotebook：ゲート数別の表示検証

`qni.show_circuit_and_state(...)` の表示を、意味のある量子回路を使って、空回路から125ゲートのQAOAまで6段階で確認するノートブックです。

量子ビット数はすべて5に固定しています。各セルを上から順に実行し、回路の横スクロール、ステップ移動、状態ベクトル表示を確認してください。

In [ ]:
from math import pi

from qni_jupyter import qni
from quri_parts.circuit import QuantumCircuit

## 検証用の量子アルゴリズムを作る関数

短い回路にはBell状態、GHZ状態、Bernstein–Vaziraniを使います。中規模回路には4ビットGrover探索、長い回路には5頂点リングのMaxCutを対象にしたQAOA Ansatzを使います。QAOAの角度は表示検証用の固定値で、最適化処理そのものは行いません。

In [ ]:
QUBIT_COUNT = 5


def build_initial_state() -> QuantumCircuit:
    return QuantumCircuit(QUBIT_COUNT)


def build_bell_state() -> QuantumCircuit:
    circuit = QuantumCircuit(QUBIT_COUNT)
    circuit.add_H_gate(0)
    circuit.add_CNOT_gate(0, 1)
    return circuit


def build_ghz_state() -> QuantumCircuit:
    circuit = QuantumCircuit(QUBIT_COUNT)
    circuit.add_H_gate(0)
    for target in range(1, QUBIT_COUNT):
        circuit.add_CNOT_gate(target - 1, target)
    return circuit


def build_bernstein_vazirani(secret: str = "1011") -> QuantumCircuit:
    if len(secret) != QUBIT_COUNT - 1 or set(secret) - {"0", "1"}:
        raise ValueError("secret must be a 4-bit string")

    ancilla = QUBIT_COUNT - 1
    circuit = QuantumCircuit(QUBIT_COUNT)
    circuit.add_X_gate(ancilla)
    for qubit in range(QUBIT_COUNT):
        circuit.add_H_gate(qubit)
    for qubit, bit in enumerate(secret):
        if bit == "1":
            circuit.add_CNOT_gate(qubit, ancilla)
    for qubit in range(QUBIT_COUNT - 1):
        circuit.add_H_gate(qubit)
    return circuit


def build_grover_28th_state_steps(iterations: int = 4):
    if iterations < 1:
        raise ValueError("iterations must be 1 or greater")

    steps = [
        [{"type": "X", "targets": [0, 1, 2, 3, 4]}],
        [{"type": "H", "targets": [0, 1, 2, 3, 4]}],
    ]
    for _ in range(iterations):
        steps.extend(
            [
                [
                    {
                        "type": "Z",
                        "targets": [0],
                        "controls": [1, 3, 4],
                        "antiControls": [2],
                    }
                ],
                [{"type": "H", "targets": [0, 1, 2, 3]}],
                [
                    {
                        "type": "X",
                        "targets": [4],
                        "controls": [0, 1, 2, 3],
                    }
                ],
                [{"type": "H", "targets": [0, 1, 2, 3]}],
            ]
        )
    return steps


def build_ring_maxcut_qaoa(depth: int) -> QuantumCircuit:
    if depth < 1:
        raise ValueError("depth must be 1 or greater")

    edges = [(qubit, (qubit + 1) % QUBIT_COUNT) for qubit in range(QUBIT_COUNT)]
    circuit = QuantumCircuit(QUBIT_COUNT)
    for qubit in range(QUBIT_COUNT):
        circuit.add_H_gate(qubit)

    for layer in range(depth):
        gamma = pi * (layer + 1) / (2 * depth + 1)
        beta = pi * (depth - layer) / (4 * depth + 2)
        for control, target in edges:
            circuit.add_CNOT_gate(control, target)
            circuit.add_RZ_gate(target, 2 * gamma)
            circuit.add_CNOT_gate(control, target)
        for qubit in range(QUBIT_COUNT):
            circuit.add_RX_gate(qubit, 2 * beta)
    return circuit


circuits = {
    "initial": build_initial_state(),
    "bell": build_bell_state(),
    "ghz": build_ghz_state(),
    "bernstein_vazirani": build_bernstein_vazirani(),
    "qaoa_depth_6": build_ring_maxcut_qaoa(depth=6),
}
grover_28th_steps = build_grover_28th_state_steps(iterations=4)
[(name, len(circuit.gates)) for name, circuit in circuits.items()] + [
    ("grover_28th_state", f"{len(grover_28th_steps)} steps / 50 gates")
]

## 1. 初期状態（0ゲート）

初期状態のままの回路です。ゲートが1つもない場合の表示を確認します。

In [ ]:
qni.show_circuit_and_state(circuits["initial"])

## 2. Bell状態（2ゲート）

`q0` と `q1` をエンタングルさせ、状態ベクトルが $|00000\rangle$ と $|00011\rangle$ の重ね合わせになることを確認します。

In [ ]:
qni.show_circuit_and_state(circuits["bell"])

## 3. 5量子ビットGHZ状態（5ゲート）

In [ ]:
qni.show_circuit_and_state(circuits["ghz"])

## 4. Bernstein–Vazirani（13ゲート）

1回のオラクル呼び出しで秘密ビット列 `1011` を取り出す回路です。

In [ ]:
qni.show_circuit_and_state(circuits["bernstein_vazirani"])

## 5. Grover探索：28番目の状態（18ステップ／50ゲート）

提示されたQni JSONと同じ列構造です。初期化の `XXXXX → HHHHH` の後、`Z + controls/anti-control → HHHH → 4-control X → HHHH` を4回繰り返します。

- **Grover反復数**：4回
- **回路ステップ数**：初期化2列＋4列×4反復＝18ステップ
- **物理ゲート数**：初期化10ゲート＋10ゲート×4反復＝50ゲート
- **増幅対象**：32状態を1から数えた28番目＝zero-based index `27`（$|11011\rangle$）
- **各反復後の対象確率**：25.830% → 60.242% → 89.694% → 99.918%

> **注意：** このJSONはzero-based index `28`（$|11100\rangle$）を増幅しません。index 28の最終確率は約0.00264%です。「状態28」をzero-based index 28の意味で使う場合は、オラクルのcontrol配置を変更する必要があります。

状態ペインが各ステップの確率表示を担うため、元JSONの `Probability5` 列は回路ステップには数えません。

In [ ]:
qni.close()
qni.open(
    steps=grover_28th_steps,
    qubit_count=QUBIT_COUNT,
    height=480,
    view="notebook",
    active_step="last",
    mode="inspect",
)

## 6. リングMaxCut QAOA・depth 6（125ゲート）

長い回路で、初期描画、横スクロール、ステップスライダー、最終状態の更新を確認します。

In [ ]:
qni.show_circuit_and_state(circuits["qaoa_depth_6"])

## 任意のQAOA depthを追加検証する

必要なら `CUSTOM_DEPTH` を変更して再実行してください。ゲート数は `5 + 20 × depth` です。

In [ ]:
CUSTOM_DEPTH = 8
custom_circuit = build_ring_maxcut_qaoa(CUSTOM_DEPTH)
print(f"depth={CUSTOM_DEPTH}, gates={len(custom_circuit.gates)}")
qni.show_circuit_and_state(custom_circuit)